In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import decoupler as dc
import os
import gseapy
import matplotlib.pyplot as plt
import squidpy as sq
import pickle
import anndata as ad
import re
from glob import glob
import copy

adata_infile = "data.h5ad"


adata = sc.read_h5ad(adata_infile)

mapping_file = "sample_metadata.csv"

# ----------------------------
# 2️⃣ Load mapping file
# ----------------------------
mapping_df = pd.read_csv(mapping_file, dtype=str)
mapping_df = mapping_df.loc[:, ~mapping_df.columns.duplicated()]

IGNORE_PREFIX_N = 0

cell_type_colors = {
    # --- Malignant epithelial ---
    'CEACAM-high tumor epithelial cells': '#6BA4F8',   # softened azure blue
    'Cycling Tumor Cells': '#F4BA63',                 # softened amber
    'Mucin-producing tumor cells': '#E59973',         # softened coral
    'Inflamed primary tumor epithelial cells': '#FF6259', # softened red

    # --- Tumor microenvironment ---
    'Complement immunosuppressive macrophages (TAMs)': '#874284',  # softened purple
    'Systemic inflammatory macrophage program (TAMs)': '#61385B',   # softened plum
    'CAFs (Cancer associated fibroblasts)': '#9A5766',              # softened burgundy
    'Pericyte-enriched endothelial cells': '#4F709D',              # softened navy

    # --- Immune cells ---
    'Cytotoxic T cells': '#998CFA',   # softened lavender
    'Plasma Cells': '#56B356',        # softened green
}


In [ ]:
"""
Full category pipeline — with primary LTS vs STS stacked bar chart added.
"""

import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG  (user configuration)
# ─────────────────────────────────────────────────────────────────────────────
IGNORE_PREFIX_N = 0

cell_types = [
    # --- Malignant epithelial ---
    'CEACAM-high tumor epithelial cells',
    'Cycling Tumor Cells',
    'Mucin-producing tumor cells',
    'Inflamed primary tumor epithelial cells',

    # --- Tumor microenvironment ---
    'Complement immunosuppressive macrophages (TAMs)',
    'Systemic inflammatory macrophage program (TAMs)',
    'CAFs (Cancer associated fibroblasts)',
    'Pericyte-enriched endothelial cells',

    # --- Immune cells ---
    'Cytotoxic T cells',
    'Plasma Cells',
]


MALIGNANT_CELL_TYPES = [
    "CEACAM-high tumor epithelial cells",
    "Cycling Tumor Cells",
    "Mucin-producing tumor cells",
    "Inflamed primary tumor epithelial cells",
]

MALIGNANT_COLORS = {
    # --- Malignant epithelial (distinct, high contrast) ---
    'CEACAM-high tumor epithelial cells': '#468DF7',        # bright azure blue
    'Cycling Tumor Cells': '#F2A93C',                      # amber orange
    'Mucin-producing tumor cells': '#DF8050', # — coral orange',       
    'Inflamed primary tumor epithelial cells': '#FF3B30', # — vivid red',   

}

OTHER_COLOR = "#CCCCCC"

SURVIVOR_PALETTE = {            
    "LTS": "#2166ac",           
    "STS": "#d6604d",           
}                               


# ----------------------------
# 1️⃣ Normalization & Mapping
# ----------------------------
def normalize_acc1(s, ignore_n=IGNORE_PREFIX_N):
    if pd.isna(s):
        return ""
    s = str(s).strip().lower()
    s = re.sub(r'[^\w\-]', '', s)
    return s[ignore_n:]


def match_sample_acc1(sample_name, mapping_df):
    sample_key = normalize_acc1(sample_name)
    matches = mapping_df[mapping_df["Acc1_key"] == sample_key]
    if not matches.empty:
        return matches.iloc[0]
    matches = mapping_df[mapping_df["Acc1_key"].apply(
        lambda ak: ak in sample_key or sample_key in ak
    )]
    if not matches.empty:
        return matches.iloc[0]
    print(f"⚠️ Sample not found in mapping: {sample_name}")
    return None


def build_sample_metadata(adata, mapping_file, sample_col="sample"):
    mapping_df = pd.read_csv(mapping_file, dtype=str)
    mapping_df = mapping_df.loc[:, ~mapping_df.columns.duplicated()]
    mapping_df["Acc1_key"]      = mapping_df["Acc1"].apply(normalize_acc1)
    mapping_df["mapped_ID_raw"] = mapping_df["Patient_ID"]

    mapping_df["OS_from_BM"] = pd.to_numeric(
        mapping_df.get("OS from BM", pd.Series()), errors="coerce"
    )
    mapping_df["survival_group"] = mapping_df["OS_from_BM"].apply(
        lambda x: "LTS" if x >= 25 else ("STS" if x <= 6 else None)
    )


    sample_map = {s: match_sample_acc1(s, mapping_df)
                  for s in adata.obs[sample_col].unique()}

    meta_rows = []
    for s, row in sample_map.items():
        if row is None:
            meta_rows.append({
                "sample":         s,
                "mapped_ID_raw":  s,
                "base_pid":       None,
                "Tumor Location": "unknown",
                "Sample ID":      "unknown",
                "survival_group": None,
             #   "her2_status":    None,
            })
        else:
            mapped_id = row["mapped_ID_raw"]
            pid_match = re.search(r"(P-\d+)", mapped_id)
            meta_rows.append({
                "sample":         s,
                "mapped_ID_raw":  mapped_id,
                "base_pid":       pid_match.group(1) if pid_match else None,
                "Tumor Location": row["Tumor Location"],
                "Sample ID":      row["Sample ID"],
                "survival_group": row.get("survival_group"),
            })
    return pd.DataFrame(meta_rows)


# ----------------------------
# 2️⃣ Compute per-sample cell type proportions
# ----------------------------
def compute_sample_proportions_per_sample(df, sample_col="sample",
                                           celltype_col="cell_type"):
    counts      = df.groupby([sample_col, celltype_col]).size().unstack(fill_value=0)
    proportions = counts.div(counts.sum(axis=1), axis=0)

    prop_df = proportions.reset_index().melt(
        id_vars=sample_col, var_name="cell_type", value_name="proportion"
    )

    total_cells = counts.sum(axis=1).reset_index()
    total_cells.columns = [sample_col, "n_cells"]
    prop_df = prop_df.merge(total_cells, on=sample_col, how="left")

    meta = df[[sample_col, "Tumor Location", "base_pid"]].drop_duplicates(
        subset=sample_col
    )
    prop_df = prop_df.merge(meta, on=sample_col, how="left")
    return prop_df


# ----------------------------
# 3️⃣ Assign final category
# ----------------------------
def assign_final_category(row):
    if row.get("survival_group") == "LTS":
        return "Long-Term Survivors"
    elif row.get("survival_group") == "STS":
        return "Short-Term Survivors"
    return None


# ----------------------------
# 4️⃣ Utility plotting
# ----------------------------
def print_sample_counts(df, category_col, label=None):
    counts = df.groupby(category_col)["sample"].nunique().reset_index(name="n_samples")
    if label:
        print(f"\n{label}:")
    print(counts.to_string(index=False))


def plot_sample_level_by_category(df, category_col, category_order=None,
                                   title="", cell_type_colors=None,
                                   use_mapped_id=True,
                                   sort_within_category=True):
    x_col  = "mapped_ID_raw" if use_mapped_id else "sample"
    counts = df.pivot_table(index=x_col, columns="cell_type",
                            values="proportion", fill_value=0)

    if category_order and sort_within_category:
        df_cat        = df[[category_col, x_col, "base_pid"]].drop_duplicates()
        cat_order_map = {k: i for i, k in enumerate(category_order)}
        df_cat["cat_rank"] = df_cat[category_col].map(cat_order_map)
        df_cat        = df_cat.sort_values(["cat_rank", "base_pid", x_col])
        counts        = counts.loc[df_cat[x_col]]

    if cell_type_colors:
        counts = counts[[c for c in cell_type_colors if c in counts.columns]]
        colors = [cell_type_colors[c] for c in counts.columns]
    else:
        colors = None

    ax = counts.plot(kind="bar", stacked=True,
                     figsize=(max(10, 0.5 * len(counts)), 6),
                     color=colors, edgecolor="black")
    ax.set_title(title)
    ax.set_ylabel("Fraction of cells")
    plt.xticks(rotation=90)
    plt.legend(title="Cell type", bbox_to_anchor=(1.02, 1),
               loc="upper left", frameon=False)
    plt.tight_layout()
    plt.show()


def plot_category_stacked_bar(df, category_col, category_order=None,
                               title="", cell_type_colors=None):
    mean_props = (df.groupby([category_col, "cell_type"])["proportion"]
                  .mean().unstack(fill_value=0))
    if category_order:
        mean_props = mean_props.reindex(category_order)
    if cell_type_colors:
        mean_props = mean_props[[c for c in cell_type_colors
                                 if c in mean_props.columns]]
    ax = mean_props.plot(
        kind="bar", stacked=True,
        figsize=(max(8, 0.8 * len(mean_props)), 4),
        color=[cell_type_colors[c] for c in mean_props.columns]
        if cell_type_colors else None,
        edgecolor="black",
        width=0.9   
    )
    ax.set_ylabel("Mean cell fraction")
    ax.set_title(title)
    plt.xticks(rotation=90)
    plt.legend(title="Cell type", bbox_to_anchor=(1.02, 1),
               loc="upper left", frameon=False)
    plt.tight_layout()
    plt.show()

import matplotlib.colors as mcolors

import matplotlib.colors as mcolors

def lighten_color(color, amount=0.5):
    """
    Lightens the given color by mixing it with white.
    amount=0 returns the original color, amount=1 returns white.
    """
    try:
        c = mcolors.to_rgb(color)
        white = (1, 1, 1)
        return tuple(c[i] + (white[i] - c[i]) * amount for i in range(3))
    except:
        return color

def plot_category_grouped_bar(df, category_col, category_order=None,
                               title="", cell_type_colors=None, 
                               lighten_amount=0.4, use_short_names=False):
    """
    Create a grouped bar plot comparing proportions across categories.
    Each cell type gets its own group of bars (one per category).
    Uses hatching pattern for the second category.
    """

    # Mapping of full names to short names
    cell_type_short_names = {
        'CEACAM-high tumor epithelial cells': 'CEACAM+ Tumor',
        'Cycling Tumor Cells': 'Cycling Tumor',
        'Mucin-producing tumor cells': 'Mucin+ Tumor',
        'Inflamed primary tumor epithelial cells': 'Inflamed Tumor',
        'Complement immunosuppressive macrophages (TAMs)': 'C1q+ TAMs',
        'Systemic inflammatory macrophage program (TAMs)': 'Inflammatory TAMs',
        'CAFs (Cancer associated fibroblasts)': 'CAFs',
        'Pericyte-enriched endothelial cells': 'Pericyte+ Endo',
        'Cytotoxic T cells': 'CD8+ T cells',
        'Plasma Cells': 'Plasma cells',
    }

    # Calculate mean proportions (UNCHANGED)
    mean_props = (
        df.groupby([category_col, "cell_type"])["proportion"]
        .mean()
        .unstack(fill_value=0)
    )

    if category_order:
        mean_props = mean_props.reindex(category_order)

    if cell_type_colors:
        mean_props = mean_props[[c for c in cell_type_colors if c in mean_props.columns]]

    # Transpose so cell types are on x-axis
    mean_props_T = mean_props.T

    fig, ax = plt.subplots(figsize=(max(8, 0.6 * len(mean_props_T)), 4))

    cell_types = mean_props_T.index.tolist()
    categories = mean_props_T.columns.tolist()

    # Optional short names
    if use_short_names:
        cell_types_display = [cell_type_short_names.get(ct, ct) for ct in cell_types]
    else:
        cell_types_display = cell_types

    x = np.arange(len(cell_types))
    width = 0.35
    multiplier = 0

    hatches = ['', '///']

    # Plot bars
    for i, category in enumerate(categories):
        offset = width * multiplier

        # ✔ ONLY CHANGE: convert to percent for display
        values = mean_props_T[category].values * 100

        colors = [cell_type_colors.get(ct, '#808080') for ct in cell_types]

        ax.bar(
            x + offset,
            values,
            width,
            label=category,
            color=colors,
            edgecolor='black',
            linewidth=1.0,
            hatch=hatches[i]
        )

        multiplier += 1

    # ── Styling ───────────────────────────────────────────────────────────────

    ax.set_xlabel('Cell type', fontsize=14, fontweight="bold")
    ax.set_ylabel('Mean cell (%)', fontsize=14, fontweight="bold")
    ax.set_title(title, fontsize=11)

    ax.set_xticks(x + width / 2)
    ax.set_xticklabels(cell_types_display, rotation=45, ha='right', fontsize=13)
    ax.tick_params(axis='y', labelsize=12)
    
    ax.legend(
        title=category_col,
        bbox_to_anchor=(1.02, 1),
        loc='upper left',
        frameon=False,
        fontsize=11
    )

    ax.set_ylim(0, 40)

    plt.tight_layout()
    return fig
# ----------------------------
# 5️⃣ Brain Met filter
# ----------------------------
def brain_met_only(df):
    return df[df["Tumor Location"] == "EAC Brain Met"]


# ----------------------------                                              
# 5b. Primary LTS vs STS stacked bar chart                                
# ----------------------------                                            
def _sort_by_dominant(prop_df_cohort,                                    
                      malignant_types=MALIGNANT_CELL_TYPES):             
    """Sort samples by dominant malignant type rank, then proportion."""  
    prop_wide = (                                                         
        prop_df_cohort                                                    
        .pivot_table(index="mapped_ID_raw", columns="cell_type",         
                     values="proportion", fill_value=0)                  
        .reset_index()                                                    
    )                                                                     
    mal_cols = [c for c in malignant_types if c in prop_wide.columns]    
    if not mal_cols:                                                      
        return prop_wide["mapped_ID_raw"].tolist(), pd.Series(dtype=str) 
                                                                         
    dominant  = prop_wide[mal_cols].idxmax(axis=1)                       
    dom_prop  = prop_wide[mal_cols].max(axis=1)                          
    type_rank = {t: i for i, t in enumerate(malignant_types)}            
                                                                         
    summary = pd.DataFrame({                                             
        "mapped_ID_raw": prop_wide["mapped_ID_raw"].values,              
        "dominant_type": dominant.values,                                
        "dominant_prop": dom_prop.values,                                
        "type_rank":     [type_rank.get(t, len(malignant_types))         
                          for t in dominant.values],                     
    }).sort_values(["type_rank", "dominant_prop"],                       
                   ascending=[True, False])                              
                                                                         
    return (summary["mapped_ID_raw"].tolist(),                           
            summary.set_index("mapped_ID_raw")["dominant_type"])         


def plot_primary_lts_vs_sts_bars(                                        
    combined_df,                                                         
    cell_type_colors=None,                                               
    malignant_types=MALIGNANT_CELL_TYPES,                                
    primary_label="Primary EAC",                                         
):                                                                       
    """                                                                  
    Stacked bar chart for PRIMARY tumours: LTS | STS.                    
    One bar per sample (mapped_ID_raw), grouped by survivor status.      
    """                                                                  
    prim_df = combined_df[                                               
        combined_df["Tumor Location"] == primary_label                  
    ].copy()                                                             
                                                                         
    prim_df = prim_df[prim_df["survival_group"].isin(["LTS", "STS"])]   
                                                                         
    if prim_df.empty:                                                    
        print("  ⚠️  No LTS/STS primary samples found — skipping plot.") 
        return                                                           
                                                                         
    # Wide proportion table                                              
    prop_wide = prim_df.pivot_table(                                     
        index="mapped_ID_raw", columns="cell_type",                      
        values="proportion", fill_value=0,                               
    )                                                                    
    all_types   = list(prop_wide.columns)                                
    other_types = sorted([c for c in all_types                          
                          if c not in malignant_types])                  
    col_order   = [c for c in malignant_types if c in all_types]        
    col_order  += other_types                                            
    prop_wide   = prop_wide[col_order]                                   
                                                                         
    colors = []                                                          
    for ct in col_order:                                                 
        if cell_type_colors and ct in cell_type_colors:                 
            colors.append(cell_type_colors[ct])                         
        elif ct in MALIGNANT_COLORS:                                     
            colors.append(MALIGNANT_COLORS[ct])                         
        else:                                                            
            colors.append(OTHER_COLOR)                                  
                                                                         
    # Sort each cohort independently                                     
    lts_df = prim_df[prim_df["survival_group"] == "LTS"]                
    sts_df = prim_df[prim_df["survival_group"] == "STS"]                
    lts_ordered, lts_dominant = _sort_by_dominant(lts_df, malignant_types) 
    sts_ordered, sts_dominant = _sort_by_dominant(sts_df, malignant_types) 
    all_samples = lts_ordered + sts_ordered                             
                                                                         
    # X positions                                                        
    GAP, BAR_W = 1.5, 0.7                                               
    x_pos = {}                                                           
    for i, s in enumerate(lts_ordered):                                 
        x_pos[s] = float(i)                                             
    lts_end = len(lts_ordered) - 1 if lts_ordered else -1              
    for i, s in enumerate(sts_ordered):                                 
        x_pos[s] = lts_end + GAP + 1.0 + float(i)                      
                                                                         
    # Draw                                                               
    fig, ax = plt.subplots(                                             
        figsize=(max(10, len(all_samples) * 0.85), 6),                  
        constrained_layout=True,                                        
    )                                                                    
    for sample in all_samples:                                          
        if sample not in prop_wide.index:                               
            continue                                                     
        xp, bottom = x_pos[sample], 0.0                                 
        for ct_val, color in zip(prop_wide.loc[sample, col_order], colors): 
            ax.bar(xp, ct_val, BAR_W, bottom=bottom,                   
                   color=color, edgecolor="white", linewidth=0.3,       
                   zorder=2)                                             
            bottom += ct_val                                            
                                                                         
    # Background shading per dominant malignant type                    
    for dom_series, ordered in [(lts_dominant, lts_ordered),            
                                 (sts_dominant, sts_ordered)]:           
        grp_slots = {}                                                   
        for s in ordered:                                               
            grp = (dom_series.loc[s] if s in dom_series.index          
                   else "other")                                        
            grp_slots.setdefault(grp, []).append(x_pos[s])             
        for grp, slots in grp_slots.items():                            
            x_min, x_max = min(slots) - 0.45, max(slots) + 0.45       
            ax.axvspan(x_min, x_max, ymin=0, ymax=1,                   
                       color=MALIGNANT_COLORS.get(grp, "#EEEEEE"),     
                       alpha=0.07, zorder=0)                            
            ax.text((x_min + x_max) / 2, 1.03, grp,                   
                    ha="center", va="bottom", fontsize=5.5,             
                    color=MALIGNANT_COLORS.get(grp, "grey"),           
                    fontweight="bold",                                   
                    transform=ax.get_xaxis_transform())                 
                                                                         
    # Cohort labels & divider                                           
    if lts_ordered:                                                     
        ax.text(np.mean([x_pos[s] for s in lts_ordered]), 1.10,        
                "Long-term survivors (LTS)", ha="center", va="bottom",  
                fontsize=8, fontweight="bold",                          
                color=SURVIVOR_PALETTE["LTS"],                          
                transform=ax.get_xaxis_transform())                     
    if sts_ordered:                                                     
        ax.text(np.mean([x_pos[s] for s in sts_ordered]), 1.10,        
                "Short-term survivors (STS)", ha="center", va="bottom", 
                fontsize=8, fontweight="bold",                          
                color=SURVIVOR_PALETTE["STS"],                          
                transform=ax.get_xaxis_transform())                     
    if lts_ordered and sts_ordered:                                     
        ax.axvline((x_pos[lts_ordered[-1]] + x_pos[sts_ordered[0]]) / 2, 
                   color="#888888", linewidth=0.8, linestyle="--",      
                   zorder=3)                                             
                                                                         
    # Axes                                                               
    ax.set_ylim(0, 1.0)                                                 
    ax.set_yticks(np.arange(0, 1.25, 0.25))                            
    ax.set_yticklabels([f"{v:.2f}" for v in np.arange(0, 1.25, 0.25)], 
                       fontsize=7)                                       
    ax.set_ylabel("Fraction of cells", fontsize=7)                      
    ax.set_xticks([x_pos[s] for s in all_samples])                     
    ax.set_xticklabels(all_samples, fontsize=7,                        
                       rotation=90, ha="right")                         
    for tick, s in zip(ax.get_xticklabels(), all_samples):             
        grp = "LTS" if s in lts_ordered else "STS"                     
        tick.set_color(SURVIVOR_PALETTE[grp])                           
    x_all = list(x_pos.values())                                        
    ax.set_xlim(min(x_all) - 0.7, max(x_all) + 0.7)                   
    ax.spines[["top", "right"]].set_visible(False)                     
                                                                         
    # Legend                                                             
    handles = [mpatches.Patch(facecolor=c, edgecolor="grey",           
                               linewidth=0.3, label=ct)                 
               for ct, c in zip(col_order, colors)]                    
    ax.legend(handles=handles, loc="upper right", fontsize=5.5,        
              frameon=False, ncol=2,                                    
              title="Cell type", title_fontsize=6)                      
                                                                         
    ax.set_title(                                                        
        "Per-Sample Cell Type Composition: Primary EAC — LTS vs STS\n" 
        "Grouped & sorted by dominant malignant population",            
        fontsize=8, pad=20,                                             
    )                                                                    
    plt.show()                                                          


# ----------------------------
# 6️⃣ Full pipeline
# ----------------------------
def run_full_category_pipeline(adata, mapping_file, cell_type_colors,
                                sample_col="sample", celltype_col="cell_type"):
    # Sample metadata
    sample_meta = build_sample_metadata(adata, mapping_file, sample_col)

    # Attach metadata to obs
    obs_df = adata.obs.copy().reset_index()
    obs_df = obs_df.merge(
        sample_meta[["sample", "Tumor Location", "base_pid"]],
        on="sample", how="left",
    )

    # Compute per-sample proportions
    prop_df = compute_sample_proportions_per_sample(obs_df)

    # Merge with full sample_meta
    cols_to_drop = [c for c in sample_meta.columns
                    if c in prop_df.columns and c != "sample"]
    combined_df = prop_df.merge(
        sample_meta.drop(columns=cols_to_drop),
        on="sample", how="left",
    )

    # mapped_ID_raw column used by new plot — ensure it exists         
    if "mapped_ID_raw" not in combined_df.columns:                        
        combined_df["mapped_ID_raw"] = combined_df["sample"]              

    combined_df["group"] = pd.Categorical(
        combined_df["Tumor Location"],
        categories=["Primary EAC", "EAC Brain Met"],
        ordered=True,
    )
    combined_df["final_category"] = combined_df.apply(
        assign_final_category, axis=1
    )

    # ---- Tumor group ----
    plot_category_stacked_bar(
        combined_df, "group",
        category_order=["Primary EAC", "EAC Brain Met"],
        title="Cell Type Composition by Tumor Group",
        cell_type_colors=cell_type_colors,
    )
    print_sample_counts(combined_df, "group", "Sample counts by tumor group")
    plot_sample_level_by_category(
        combined_df, "group",
        category_order=["Primary EAC", "EAC Brain Met"],
        title="Per-Sample Cell Type Composition by Tumor Group",
        cell_type_colors=cell_type_colors,
        use_mapped_id=True,
    )

        # If you want to compare "Primary EAC" vs "EAC Brain Met"
    plot_category_grouped_bar(
        combined_df, "group",
        category_order=["Primary EAC", "EAC Brain Met"],
        title="Grouped bar\nCell type comparisons per cell type",
        use_short_names=True, 
        cell_type_colors=cell_type_colors
    )

    # ---- Survival group (Brain Met only) ----
    df_surv = brain_met_only(combined_df.dropna(subset=["survival_group"]))
    print_sample_counts(df_surv, "survival_group",
                        "Sample counts by survival group (Brain Met)")
    plot_category_stacked_bar(
        df_surv, "survival_group",
        category_order=["LTS", "STS"],
        title="Cell Type Composition: Long vs Short-Term Survivors (Brain Met)",
        cell_type_colors=cell_type_colors,
    )
    plot_sample_level_by_category(
        df_surv, "survival_group",
        category_order=["LTS", "STS"],
        title="Per-Sample Cell Type Composition: LTS vs STS (Brain Met)",
        cell_type_colors=cell_type_colors,
        use_mapped_id=True,
    )

    plot_category_grouped_bar(
        df_surv, "survival_group",
        category_order=["LTS", "STS"],
        title="Cell Type Composition: Long vs Short-Term Survivors (Brain Met)", 
        use_short_names=True, 
        cell_type_colors=cell_type_colors
    )

    # ---- Survival group (Primary only) ────────────────────────────────  ← NEW
    df_prim_surv = combined_df[                                           
        (combined_df["Tumor Location"] == "Primary EAC") &               
        combined_df["survival_group"].isin(["LTS", "STS"])               
    ].copy()                                                              
    print("\nPlotting Primary EAC: LTS vs STS …")                        
    print_sample_counts(df_prim_surv, "survival_group",                  
                        "Sample counts by survival group (Primary EAC)") 
    plot_category_stacked_bar(                                            
        df_prim_surv, "survival_group",                                  
        category_order=["LTS", "STS"],                                   
        title="Cell Type Composition: LTS vs STS (Primary EAC)",        
        cell_type_colors=cell_type_colors,                               
    )                                                                     
    plot_primary_lts_vs_sts_bars(                                         
        combined_df,                                                      
        cell_type_colors=cell_type_colors,                               
    )      

    plot_category_grouped_bar(
        df_prim_surv, "survival_group",                                  
        category_order=["LTS", "STS"],                                   
        title="Cell Type Composition: LTS vs STS (Primary EAC)",  
        use_short_names=True, 
        cell_type_colors=cell_type_colors,                               
    )    


    return combined_df
    return df_surv

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.ticker import MultipleLocator

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
output_dir = "primary-BM-celltype-comparison"
os.makedirs(output_dir, exist_ok=True)

GROUP_PRIMARY = "Primary EAC"
GROUP_MET     = "EAC Brain Met"



MALIGNANT_CELL_TYPES = [
    "CEACAM-high tumor epithelial cells",
    "Cycling Tumor Cells",
    "Mucin-producing tumor cells",
    "Inflamed primary tumor epithelial cells",
]

MALIGNANT_COLORS = {
    # --- Malignant epithelial (distinct, high contrast) ---
    'CEACAM-high tumor epithelial cells': '#468DF7',        # bright azure blue
    'Cycling Tumor Cells': '#F2A93C',                      # amber orange
    'Mucin-producing tumor cells': '#DF8050', # — coral orange',       
    'Inflamed primary tumor epithelial cells': '#FF3B30', # — vivid red',   

}

OTHER_COLOR = "#CCCCCC"

LABEL_FS = 7
TITLE_FS = 7.5
TICK_FS  = 6.5


# ═════════════════════════════════════════════════════════════════════════════
# 1.  GET PAIRED PATIENTS
# ═════════════════════════════════════════════════════════════════════════════
def get_paired_primary_brain_met(df,
                                  primary_label=GROUP_PRIMARY,
                                  met_label=GROUP_MET):
    """Keep only patients that have both a primary and at least one brain met."""
    counts = (
        df.groupby(["base_pid", "Tumor Location"])["sample"]
        .nunique()
        .unstack(fill_value=0)
    )
    paired_pids = counts[
        (counts.get(primary_label, pd.Series(0, index=counts.index)) > 0) &
        (counts.get(met_label,     pd.Series(0, index=counts.index)) > 0)
    ].index
    return df[df["base_pid"].isin(paired_pids)].copy()


# ═════════════════════════════════════════════════════════════════════════════
# 2.  COMPUTE PER-SAMPLE PROPORTIONS
# ═════════════════════════════════════════════════════════════════════════════
def compute_sample_proportions_per_sample(df,
                                           sample_col="sample",
                                           celltype_col="cell_type"):
    """
    Returns long-format DataFrame:
        sample | cell_type | proportion | n_cells | Tumor Location | base_pid
    n_cells = total cells in that sample (used for picking largest sample).
    """
    counts = (
        df.groupby([sample_col, celltype_col])
        .size()
        .unstack(fill_value=0)
    )
    proportions = counts.div(counts.sum(axis=1), axis=0)

    prop_df = (
        proportions.reset_index()
        .melt(id_vars=sample_col, var_name="cell_type", value_name="proportion")
    )

    # Total cells per sample
    total_cells = counts.sum(axis=1).reset_index()
    total_cells.columns = [sample_col, "n_cells"]
    prop_df = prop_df.merge(total_cells, on=sample_col, how="left")

    # Re-attach metadata
    meta = (
        df[[sample_col, "Tumor Location", "base_pid"]]
        .drop_duplicates(subset=sample_col)
    )
    prop_df = prop_df.merge(meta, on=sample_col, how="left")

    return prop_df


# ═════════════════════════════════════════════════════════════════════════════
# 3.  SELECT ONE PRIMARY + ONE MET PER PATIENT (largest by cell count)
# ═════════════════════════════════════════════════════════════════════════════
def select_largest_sample(prop_df, pid, location,
                           primary_label=GROUP_PRIMARY,
                           met_label=GROUP_MET):
    """Return the sample name with the most cells for a given patient + location."""
    subset = prop_df[
        (prop_df["base_pid"] == pid) &
        (prop_df["Tumor Location"] == location)
    ][["sample", "n_cells"]].drop_duplicates(subset="sample")
    if subset.empty:
        return None
    return subset.loc[subset["n_cells"].idxmax(), "sample"]


def get_one_primary_one_met(prop_df,
                             primary_label=GROUP_PRIMARY,
                             met_label=GROUP_MET):
    """
    For each paired patient keep only:
        - the primary sample with the most cells
        - the brain met sample with the most cells
    Returns a filtered prop_df containing only those samples.
    """
    paired_pids = (
        prop_df[prop_df["Tumor Location"].isin([primary_label, met_label])]
        .groupby(["base_pid", "Tumor Location"])["sample"]
        .nunique()
        .unstack(fill_value=0)
        .pipe(lambda c: c[
            (c.get(primary_label, pd.Series(0, index=c.index)) > 0) &
            (c.get(met_label,     pd.Series(0, index=c.index)) > 0)
        ])
        .index
    )

    selected_samples = []
    for pid in paired_pids:
        ps = select_largest_sample(prop_df, pid, primary_label)
        ms = select_largest_sample(prop_df, pid, met_label)
        if ps and ms:
            selected_samples.extend([ps, ms])

    return prop_df[prop_df["sample"].isin(selected_samples)].copy()



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def plot_paired_boxplots_and_save_table(prop_df, output_tsv="primary_brain_met_proportions.tsv"):
    """
    Paired boxplots Primary vs Brain Met with lines connecting each patient.
    Uses original blue/orange colors. Saves the exact per-sample proportions used in plots.
    """
    group_order = ["Primary EAC", "EAC Brain Met"]
    palette = {"Primary EAC":"black",  # blue
               "EAC Brain Met":"black"} # orange
    
    df = prop_df[prop_df["Tumor Location"].isin(group_order)].copy()
    
    # Only patients with at least one primary and one brain met
    counts = df.groupby(["base_pid", "Tumor Location"])["sample"].nunique().unstack(fill_value=0)
    paired_pids = counts[(counts.get("Primary EAC",0)>0) & (counts.get("EAC Brain Met",0)>0)].index
    df = df[df["base_pid"].isin(paired_pids)].copy()
    
    cell_types = df["cell_type"].unique()
    
    # Prepare list to collect proportions for saving
    saved_rows = []

    for ct in cell_types:
        df_ct = df[df["cell_type"]==ct]
        plt.figure(figsize=(4,4))
        
        # Boxplot with fixed blue/orange palette
        sns.boxplot(
            data=df_ct,
            x="Tumor Location",
            y="proportion",
            hue="Tumor Location",
            hue_order=group_order,
            palette=palette,
            dodge=False,
            fliersize=0,
            linewidth=1
        )
        
        # Overlay individual points
        sns.stripplot(
            data=df_ct,
            x="Tumor Location",
            y="proportion",
            hue="Tumor Location",
            hue_order=group_order,
            dodge=False,
            color="black",
            alpha=0.6,
            marker="o",
            linewidth=0.5,
            size=5
        )
        
        # Remove duplicate legend
        plt.legend([],[], frameon=False)
        
        # Draw lines connecting primary → brain met
        for pid in df_ct["base_pid"].unique():
            patient_df = df_ct[df_ct["base_pid"]==pid]
            primary_vals = patient_df[patient_df["Tumor Location"]=="Primary EAC"]["proportion"].values
            brain_met_vals = patient_df[patient_df["Tumor Location"]=="EAC Brain Met"]["proportion"].values
            for pv in primary_vals:
                for bv in brain_met_vals:
                    plt.plot([0,1], [pv,bv], color="grey", linestyle="--", alpha=0.5)
            
            # Save the mean for this patient & cell type (exactly what was plotted)
            saved_rows.append({
                "base_pid": pid,
                "cell_type": ct,
                "primary_proportion": primary_vals.mean() if len(primary_vals)>0 else None,
                "brain_met_proportion": brain_met_vals.mean() if len(brain_met_vals)>0 else None
            })
        
        # Dynamically set y-axis max based on largest value
        max_val = df_ct["proportion"].max()
        plt.ylim(0, min(max_val*1.1, 1.0))  # Add 10% padding, cap at 1
        
        plt.title(f" {ct}")
        plt.ylabel("Fraction of cells")
        plt.xticks(rotation=0)
        plt.tight_layout()
        plt.show()
    
    # Save the table
    proportions_df = pd.DataFrame(saved_rows)
    proportions_df.to_csv(output_tsv, sep="\t", index=False)
    print(f"✅ Saved Primary vs Brain Met proportions to {output_tsv}")
    
    return proportions_df
proportions_df = plot_paired_boxplots_and_save_table(combined_df)



##can be update for different comparisons

"""
Cell-type proportion plots — Primary vs Brain Metastasis
─────────────────────────────────────────────────────────
Figure 1  →  Per-sample grid  (one panel per patient, all cell types)
Figure 2  →  Cohort-level grid (one panel per cell type, all patients)
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats

# ── Data ───────────────────────────────────────────────────────────────────────
df = pd.read_csv(
    "primary_brain_met_proportions.tsv",
    sep="\t",
)

samples    = sorted(df["base_pid"].unique())
cell_types = sorted(df["cell_type"].unique())
n_samples  = len(samples)
n_types    = len(cell_types)

# ── Shared palette ─────────────────────────────────────────────────────────────
ct_colors  = dict(zip(cell_types, plt.cm.tab10(np.linspace(0, 1, n_types))))
cell_type_colors = {
    # Malignant epithelial
    "CEACAM-high tumor epithelial cells":       "#6BA4F8",
    "Cycling Tumor Cells":                      "#F4BA63",
    "Mucin-producing tumor cells":              "#E59973",
    "Inflamed primary tumor epithelial cells":  "#FF6259",
    # Tumour microenvironment
    "Complement immunosuppressive macrophages (TAMs)": "#998CFA",
    "Systemic inflammatory macrophage program (TAMs)": "#61385B",
    "CAFs (Cancer associated fibroblasts)":     "#9A5766",
    "Pericyte-enriched endothelial cells":      "#4F709D",
    # Immune
    "Cytotoxic T cells":                        "#874284",
    "Plasma Cells":                             "#56B356",
}


pid_colors = dict(zip(samples,    plt.cm.tab20(np.linspace(0, 1, n_samples))))

# ── Shared style helpers ───────────────────────────────────────────────────────
LABEL_FS   = 14
TITLE_FS   = 14
TICK_FS    = 12
DOT_S      = 26
LINE_ALPHA = 0.55
LINE_LW    = 0.9

def style_ax(ax):
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Primary", "Brain Met"], fontsize=TICK_FS)
    ax.set_xlim(-0.45, 1.45)
    ax.tick_params(axis="y", labelsize=TICK_FS)

def scatter_pairs(ax, primary, met, line_color, dot_ec="black"):
    n = len(primary)
    for i in range(n):
        col = "#d62728" if primary[i] > met[i] else line_color
        ax.plot([0, 1], [primary[i], met[i]],
                color=col, alpha=LINE_ALPHA, linewidth=LINE_LW, zorder=1)
    ax.scatter(np.zeros(n), primary, color="white", edgecolors=dot_ec,
               s=DOT_S, zorder=3, linewidths=0.7)
    ax.scatter(np.ones(n),  met,     color="white", edgecolors=dot_ec,
               s=DOT_S, zorder=3, linewidths=0.7)

def wilcoxon_p(a, b):
    a, b = np.asarray(a), np.asarray(b)
    if len(a) < 4 or np.all(a == b):
        _, p = stats.ttest_rel(a, b)
    else:
        _, p = stats.wilcoxon(a, b, zero_method="wilcox", alternative="two-sided")
    return p

def fmt_p(p):
    if p < 0.001: return "P < 0.001 ***"
    if p < 0.01:  return f"P = {p:.3f} **"
    if p < 0.05:  return f"P = {p:.3f} *"
    return f"P = {p:.2f} ns"


# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 1 — Per-sample  (one panel per patient)
# ══════════════════════════════════════════════════════════════════════════════
ncols = 5
nrows = int(np.ceil(n_samples / ncols))

fig1, axes1 = plt.subplots(
    nrows, ncols,
    figsize=(ncols * 3.2, nrows * 3.6),
    constrained_layout=True,
)
axes1_flat = axes1.flatten()

for idx, pid in enumerate(samples):
    ax  = axes1_flat[idx]
    sub = df[df["base_pid"] == pid].set_index("cell_type")

    primary_vals = []
    met_vals     = []
    colors_used  = []

    for ct in cell_types:
        if ct in sub.index:
            pv = sub.loc[ct, "primary_proportion"]
            mv = sub.loc[ct, "brain_met_proportion"]
            primary_vals.append(pv)
            met_vals.append(mv)
            # line color = cell-type color; red if decreasing
            lc = "#d62728" if pv > mv else ct_colors[ct]
            ax.plot([0, 1], [pv, mv],
                    color=lc, alpha=LINE_ALPHA, linewidth=LINE_LW, zorder=1)

    ax.scatter(np.zeros(len(primary_vals)), primary_vals,
               color="white", edgecolors="black", s=DOT_S, zorder=3, linewidths=0.7)
    ax.scatter(np.ones(len(met_vals)),      met_vals,
               color="white", edgecolors="black", s=DOT_S, zorder=3, linewidths=0.7)

    # per-sample paired p-value across all cell types
    if len(primary_vals) >= 2:
        p = wilcoxon_p(primary_vals, met_vals)
    else:
        p = np.nan

    ax.set_title(f"{pid}\n{fmt_p(p)}", fontsize=TITLE_FS, pad=3)
    ax.set_ylabel("Proportion", fontsize=LABEL_FS)
    style_ax(ax)

# hide unused axes
for ax in axes1_flat[n_samples:]:
    ax.set_visible(False)

# legend
legend_handles = [
    mpatches.Patch(facecolor=ct_colors[ct], label=ct, alpha=0.85)
    for ct in cell_types
]
fig1.legend(
    handles=legend_handles,
    fontsize=6,
    loc="lower right",
    ncol=2,
    frameon=False,
    bbox_to_anchor=(1.01, 0.0),
    title="Cell type",
    title_fontsize=7,
)
fig1.suptitle(
    "Per-sample cell-type proportions — Primary vs Brain Met\n"
    "(Wilcoxon signed-rank across all cell types per patient; red lines = decrease in met)",
    fontsize=10, fontweight="bold",
)

fig1.savefig("fig1_Per-sample cell-type proportions.png",
             bbox_inches="tight", dpi=200)
fig1.savefig("fig1_Per-sample cell-type proportions.pdf",
             bbox_inches="tight")
print("Figure 1 saved.")


# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 2 — Cohort-level paired lines  (one panel per cell type, no SEM bars)
# ══════════════════════════════════════════════════════════════════════════════
ncols2 = 4
nrows2 = int(np.ceil(n_types / ncols2))

fig2, axes2 = plt.subplots(
    nrows2, ncols2,
    figsize=(ncols2 * 3.4, nrows2 * 3.8),
    constrained_layout=True,
)
axes2_flat = axes2.flatten()

cohort_pvals = {}

for idx, ct in enumerate(cell_types):
    ax  = axes2_flat[idx]
    sub = df[df["cell_type"] == ct]
    primary = sub["primary_proportion"].values
    met     = sub["brain_met_proportion"].values

    scatter_pairs(ax, primary, met, line_color=ct_colors[ct])

    p = wilcoxon_p(primary, met)
    cohort_pvals[ct] = p

    # annotate individual sample IDs lightly
    for i, pid in enumerate(sub["base_pid"].values):
        ax.text(1.07, met[i], pid, fontsize=4.5,
                va="center", color="gray", alpha=0.7)

    short_ct = ct.replace("(Cancer associated fibroblasts)", "(CAFs)")
    ax.set_title(f"{short_ct}\n{fmt_p(p)}", fontsize=TITLE_FS, pad=3)
    ax.set_ylabel("Proportion", fontsize=LABEL_FS)
    style_ax(ax)

for ax in axes2_flat[n_types:]:
    ax.set_visible(False)

fig2.suptitle(
    "Cohort-level cell-type proportions — Primary vs Brain Met\n"
    "(Wilcoxon signed-rank per cell type; red lines = decrease in met)",
    fontsize=10, fontweight="bold",
)
fig2.savefig("fig2_Cohort-level cell-type proportions.png",
             bbox_inches="tight", dpi=200)
fig2.savefig("fig2_Cohort-level cell-type proportions.pdf",
             bbox_inches="tight")
print("Figure 2 saved.")


# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 3 — Mean ± SEM summary  (one panel per cell type, no individual lines)
# ══════════════════════════════════════════════════════════════════════════════
fig3, axes3 = plt.subplots(
    nrows2, ncols2,
    figsize=(ncols2 * 3.0, nrows2 * 3.4),
    constrained_layout=True,
)
axes3_flat = axes3.flatten()

for idx, ct in enumerate(cell_types):
    ax  = axes3_flat[idx]
    sub = df[df["cell_type"] == ct]
    primary = sub["primary_proportion"].values
    met     = sub["brain_met_proportion"].values
    color   = ct_colors[ct]
    p       = cohort_pvals[ct]

    means = [np.mean(primary), np.mean(met)]
    sems  = [stats.sem(primary), stats.sem(met)]

    # shaded area between the two means
    ax.fill_between([0, 1], [means[0] - sems[0], means[1] - sems[1]],
                             [means[0] + sems[0], means[1] + sems[1]],
                    color=color, alpha=0.18, zorder=1)

    # connecting line between means
    ax.plot([0, 1], means, color=color, linewidth=2.0, zorder=2)

    # error bars
    for xi, m, se in zip([0, 1], means, sems):
        ax.errorbar(xi, m, yerr=se,
                    fmt="o", color=color,
                    markerfacecolor="white", markeredgewidth=1.5,
                    markersize=8, capsize=5, capthick=1.5,
                    linewidth=1.5, zorder=3)

    # delta annotation (arrow + Δ value)
    delta = means[1] - means[0]
    sign  = "+" if delta >= 0 else "−"
    ax.annotate(
        f"Δ = {sign}{abs(delta):.3f}",
        xy=(0.5, max(means) + max(sems) * 1.3),
        ha="center", fontsize=6.5, color="dimgray",
    )

    short_ct = ct.replace("(Cancer associated fibroblasts)", "(CAFs)")
    ax.set_title(f"{short_ct}\n{fmt_p(p)}", fontsize=TITLE_FS, pad=3)
    ax.set_ylabel("Mean proportion", fontsize=LABEL_FS)
    style_ax(ax)

for ax in axes3_flat[n_types:]:
    ax.set_visible(False)

fig3.suptitle(
    "Cohort-level Mean ± SEM — Primary vs Brain Met\n"
    "(Wilcoxon signed-rank p-values; Δ = met − primary)",
    fontsize=10, fontweight="bold",
)
fig3.savefig("fig3_cohort_mean_sem.png",
             bbox_inches="tight", dpi=200)
fig3.savefig("fig3_cohort_mean_sem.pdf",
             bbox_inches="tight")
print("Figure 3 saved.")

# ── P-value summary ────────────────────────────────────────────────────────────
print("\n── Cohort-level Wilcoxon p-values ──────────────────────────────────")
print(f"{'Cell type':<52} {'P-value':>9}  Sig")
print("─" * 68)
for ct in cell_types:
    p   = cohort_pvals[ct]
    sig = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "ns"))
    print(f"{ct:<52} {p:>9.4f}  {sig}")
    

# Figure 4 — Selected cell types mean % ± SEM (Primary vs Brain Met)

import matplotlib.ticker as mticker

cell_types_of_interest = [
    "CEACAM-high tumor epithelial cells",
    # "Cycling Tumor Cells",
    # "Mucin-producing tumor cells",
    # "Inflamed primary tumor epithelial cells",
    "Cytotoxic T cells",
    "Plasma Cells",
    "Inflamed primary tumor epithelial cells",
    # "Systemic inflammatory macrophage program (TAMs)"
]

selected_cell_types = [ct for ct in cell_types_of_interest if ct in cell_types]
n_types_sel = len(selected_cell_types)

ncols4 = 3
nrows4 = int(np.ceil(n_types_sel / ncols4))

fig4, axes4 = plt.subplots(
    nrows4, ncols4,
    figsize=(ncols4 * 3.0, nrows4 * 3.8),
    constrained_layout=True,
)

axes4_flat = axes4.flatten()

for idx, ct in enumerate(selected_cell_types):
    ax = axes4_flat[idx]

    sub = df[df["cell_type"] == ct]

    # Convert proportions to percentages
    primary = sub["primary_proportion"].values * 100
    met     = sub["brain_met_proportion"].values * 100

    color = cell_type_colors[ct]
    p = cohort_pvals[ct]

    # Mean and SEM in percent units
    means = [np.mean(primary), np.mean(met)]
    sems  = [stats.sem(primary), stats.sem(met)]

    # Shaded SEM band between means
    ax.fill_between(
        [0, 1],
        [means[0] - sems[0], means[1] - sems[1]],
        [means[0] + sems[0], means[1] + sems[1]],
        color=color,
        alpha=0.18,
        zorder=1,
    )

    # Connecting line
    ax.plot(
        [0, 1],
        means,
        color=color,
        linewidth=2.0,
        zorder=2,
    )

    # Mean ± SEM points
    for xi, m, se in zip([0, 1], means, sems):
        ax.errorbar(
            xi,
            m,
            yerr=se,
            fmt="o",
            color=color,
            markerfacecolor="white",
            markeredgewidth=1.5,
            markersize=8,
            capsize=5,
            capthick=1.5,
            linewidth=1.5,
            zorder=3,
        )

    short_ct = ct.replace("(Cancer associated fibroblasts)", "(CAFs)")

    ax.set_title(
        f"{short_ct}\n{fmt_p(p)}",
        fontsize=TITLE_FS,
        pad=3,
    )

    # Updated y-axis label
    ax.set_ylabel("Mean %", fontsize=LABEL_FS, fontweight="bold")

    # Format ticks as percentages
    ax.yaxis.set_major_formatter(mticker.PercentFormatter())

    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Primary", "Brain Met"], fontsize=LABEL_FS, fontweight="bold")

    style_ax(ax)

# Hide unused subplots
for ax in axes4_flat[n_types_sel:]:
    ax.set_visible(False)

fig4.suptitle(
    "Selected Cell Types Mean % ± SEM — Primary vs Brain Met\n"
    "(Wilcoxon signed-rank p-values; Δ = met − primary)",
    fontsize=10,
    fontweight="bold",
)

fig4.savefig(
    "fig4_cohort_mean_percent_sem-select-celltypes.png",
    bbox_inches="tight",
    dpi=200,
)

fig4.savefig(
    "fig4_cohort_mean_percent_sem-select-celltypes.pdf",
    bbox_inches="tight",
)

print("Figure 4 saved.")


# ── P-value summary ────────────────────────────────────────────────────────────
print("\n── Cohort-level Wilcoxon p-values ──────────────────────────────────")
print(f"{'Cell type':<52} {'P-value':>9}  Sig")
print("─" * 68)
for ct in cell_types_of_interest:
    p   = cohort_pvals[ct]
    sig = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "ns"))
    print(f"{ct:<52} {p:>9.4f}  {sig}")

    

In [ ]:
##LTS vs STS comparative analysis, primary to brain met cell type proportions

"""
Cell-type proportion plots — Primary vs Brain Metastasis
─────────────────────────────────────────────────────────
Figure 1  →  Per-sample grid         (one panel per patient, all cell types)
Figure 2  →  Survivor-stratified     (one panel per cell type, LT vs ST paired lines)
Figure 3  →  Mean ± SEM summary      (one panel per cell type, LT vs ST overlaid)
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
from scipy import stats

# ── Data ───────────────────────────────────────────────────────────────────────
df = pd.read_csv(
    "primary_brain_met_proportions.tsv",
    sep="\t",
)

# ── Survivor classification ────────────────────────────────────────────────────
def classify_survivor(pid):
    try:
        n = int(pid.split("-")[1])
    except (IndexError, ValueError):
        return "unclassified"
    if 1 <= n <= 14:
        return "Long-term"
    if 45 <= n <= 65:
        return "Short-term"
    return "unclassified"

df["survivor_group"] = df["base_pid"].apply(classify_survivor)

samples    = sorted(df["base_pid"].unique())
cell_types = sorted(df["cell_type"].unique())
n_samples  = len(samples)
n_types    = len(cell_types)

groups       = ["Long-term", "Short-term"]
group_colors = {"Long-term": "#2166AC", "Short-term": "#292929"}
ct_colors    = dict(zip(cell_types, plt.cm.tab10(np.linspace(0, 1, n_types))))


# ── Report groups ──────────────────────────────────────────────────────────────
for g in groups:
    pids = sorted(df[df["survivor_group"] == g]["base_pid"].unique())
    print(f"{g} (n={len(pids)}): {', '.join(pids)}")
unc = sorted(df[df["survivor_group"] == "unclassified"]["base_pid"].unique())
print(f"Unclassified (n={len(unc)}): {', '.join(unc)}")

# ── Shared style ───────────────────────────────────────────────────────────────
LABEL_FS   = 14
TITLE_FS   = 14
TICK_FS    = 14
DOT_S      = 26
LINE_ALPHA = 0.55
LINE_LW    = 0.9

def style_ax(ax):
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Primary", "Brain Met"], fontsize=TICK_FS)
    ax.set_xlim(-0.45, 1.45)
    ax.tick_params(axis="y", labelsize=TICK_FS)

def wilcoxon_p(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    if len(a) < 4 or np.allclose(a, b):
        _, p = stats.ttest_rel(a, b)
    else:
        _, p = stats.wilcoxon(a, b, zero_method="wilcox", alternative="two-sided")
    return p

def fmt_p(p):
    if np.isnan(p):  return "P = n/a"
    if p < 0.001:    return "P < 0.001 ***"
    if p < 0.01:     return f"P = {p:.3f} **"
    if p < 0.05:     return f"P = {p:.3f} *"
    return f"P = {p:.2f} ns"

def short_name(ct):
    return ct.replace("(Cancer associated fibroblasts)", "(CAFs)")


# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 1 — Per-sample  (one panel per patient, lines colored by cell type)
# ══════════════════════════════════════════════════════════════════════════════
ncols1 = 5
nrows1 = int(np.ceil(n_samples / ncols1))

fig1, axes1 = plt.subplots(nrows1, ncols1,
                            figsize=(ncols1 * 3.2, nrows1 * 3.6),
                            constrained_layout=True)
axes1_flat = axes1.flatten()

for idx, pid in enumerate(samples):
    ax    = axes1_flat[idx]
    sub   = df[df["base_pid"] == pid].set_index("cell_type")
    group = df[df["base_pid"] == pid]["survivor_group"].iloc[0]
    gc    = group_colors.get(group, "gray")

    primary_vals, met_vals = [], []
    for ct in cell_types:
        if ct in sub.index:
            pv = sub.loc[ct, "primary_proportion"]
            mv = sub.loc[ct, "brain_met_proportion"]
            primary_vals.append(pv)
            met_vals.append(mv)
            lc = "#d62728" if pv > mv else ct_colors[ct]
            ax.plot([0, 1], [pv, mv],
                    color=lc, alpha=LINE_ALPHA, linewidth=LINE_LW, zorder=1)

    ax.scatter(np.zeros(len(primary_vals)), primary_vals,
               color="white", edgecolors="black", s=DOT_S, zorder=3, linewidths=0.7)
    ax.scatter(np.ones(len(met_vals)), met_vals,
               color="white", edgecolors="black", s=DOT_S, zorder=3, linewidths=0.7)

    p = wilcoxon_p(primary_vals, met_vals) if len(primary_vals) >= 4 else np.nan
    group_label = f"[{group[:2]}]" if group != "unclassified" else ""
    ax.set_title(f"{pid} {group_label}\n{fmt_p(p)}", fontsize=TITLE_FS, pad=3,
                 color=gc if group != "unclassified" else "black")
    ax.set_ylabel("Proportion", fontsize=LABEL_FS)
    style_ax(ax)

for ax in axes1_flat[n_samples:]:
    ax.set_visible(False)

legend_handles = [mpatches.Patch(facecolor=ct_colors[ct], label=ct, alpha=0.85)
                  for ct in cell_types]
fig1.legend(handles=legend_handles, fontsize=6, loc="lower right",
            ncol=2, frameon=False, bbox_to_anchor=(1.01, 0.0),
            title="Cell type", title_fontsize=7)
fig1.suptitle(
    "Per-sample cell-type proportions — Primary vs Brain Met\n"
    "[Lo] = Long-term survivor  |  [Sh] = Short-term survivor  |  red = decrease in met",
    fontsize=10, fontweight="bold")

fig1.savefig("fig1_per_sample-LTSvSTS.png", bbox_inches="tight", dpi=200)
fig1.savefig("fig1_per_sample-LTSvSTS.pdf", bbox_inches="tight")
print("Figure 1 saved.")


# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 2 — Survivor-stratified paired lines
#            Two sub-columns per cell type: [Long-term | Short-term]
# ══════════════════════════════════════════════════════════════════════════════
ncols2 = 4
nrows2 = int(np.ceil(n_types / ncols2))

fig2, axes2 = plt.subplots(
    nrows2, ncols2 * 2,
    figsize=(ncols2 * 2 * 2.8, nrows2 * 3.8),
    constrained_layout=True,
)

cohort_pvals = {ct: {} for ct in cell_types}

for idx, ct in enumerate(cell_types):
    row = idx // ncols2
    col = (idx % ncols2) * 2   # left = Long-term, right = Short-term

    for g_idx, group in enumerate(groups):
        ax  = axes2[row, col + g_idx]
        sub = df[(df["cell_type"] == ct) & (df["survivor_group"] == group)]
        gc  = group_colors[group]

        if len(sub) == 0:
            ax.set_visible(False)
            continue

        primary = sub["primary_proportion"].values
        met     = sub["brain_met_proportion"].values

        for i in range(len(primary)):
            lc = "#d62728" if primary[i] > met[i] else gc
            ax.plot([0, 1], [primary[i], met[i]],
                    color=lc, alpha=0.6, linewidth=1.0, zorder=1)

        ax.scatter(np.zeros(len(primary)), primary,
                   color="white", edgecolors="black", s=DOT_S, zorder=3, linewidths=0.7)
        ax.scatter(np.ones(len(met)), met,
                   color="white", edgecolors="black", s=DOT_S, zorder=3, linewidths=0.7)

        for i, pid in enumerate(sub["base_pid"].values):
            ax.text(1.07, met[i], pid, fontsize=4.2,
                    va="center", color="gray", alpha=0.75)

        p = wilcoxon_p(primary, met) if len(primary) >= 4 else np.nan
        cohort_pvals[ct][group] = p

        ax.set_title(
            f"{short_name(ct)}\n{group} (n={len(primary)})\n{fmt_p(p)}",
            fontsize=TITLE_FS - 0.5, pad=3, color=gc, fontweight="bold")
        ax.set_ylabel("Proportion", fontsize=LABEL_FS)
        ax.set_facecolor((*plt.matplotlib.colors.to_rgb(gc), 0.04))
        style_ax(ax)

# hide trailing axes
if n_types % ncols2 != 0:
    last = (n_types % ncols2) * 2
    for c in range(last, ncols2 * 2):
        axes2[-1, c].set_visible(False)

leg2 = [mlines.Line2D([], [], color=group_colors[g], linewidth=2.5, label=g)
        for g in groups] + \
       [mlines.Line2D([], [], color="#d62728", linewidth=2.0,
                      linestyle="--", label="Decrease in met")]
fig2.legend(handles=leg2, fontsize=8, loc="lower right",
            frameon=False, bbox_to_anchor=(1.0, -0.01),
            title="Survivor group", title_fontsize=8.5)
fig2.suptitle(
    "Survivor-stratified paired lines — Primary vs Brain Met\n"
    "Long-term (P-01–14) vs Short-term (P-45–65)  |  Wilcoxon signed-rank  |  red = decrease in met",
    fontsize=10, fontweight="bold")

fig2.savefig("fig2_cohort_paired_lines-LTSvSTS.png",
             bbox_inches="tight", dpi=200)
fig2.savefig("fig2_cohort_paired_lines-LTSvSTS.pdf",
             bbox_inches="tight")
print("Figure 2 saved.")


# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 3 — Mean ± SEM  (both groups overlaid, one panel per cell type)
# ══════════════════════════════════════════════════════════════════════════════
fig3, axes3 = plt.subplots(
    nrows2, ncols2,
    figsize=(ncols2 * 3.2, nrows2 * 3.6),
    constrained_layout=True,
)
axes3_flat = axes3.flatten()

for idx, ct in enumerate(cell_types):
    ax = axes3_flat[idx]

    y_ann_base = 0.0
    for g_idx, group in enumerate(groups):
        sub = df[(df["cell_type"] == ct) & (df["survivor_group"] == group)]
        if len(sub) < 2:
            continue

        primary = sub["primary_proportion"].values
        met     = sub["brain_met_proportion"].values
        gc      = group_colors[group]
        p       = cohort_pvals[ct].get(group, np.nan)

        means = [np.mean(primary), np.mean(met)]
        sems  = [stats.sem(primary), stats.sem(met)]

        ax.fill_between(
            [0, 1],
            [means[0] - sems[0], means[1] - sems[1]],
            [means[0] + sems[0], means[1] + sems[1]],
            color=gc, alpha=0.15, zorder=1)
        ax.plot([0, 1], means, color=gc, linewidth=2.2,
                zorder=2, label=f"{group} (n={len(primary)})")
        for xi, m, se in zip([0, 1], means, sems):
            ax.errorbar(xi, m, yerr=se,
                        fmt="o", color=gc,
                        markerfacecolor="white", markeredgewidth=1.6,
                        markersize=8, capsize=5, capthick=1.5,
                        linewidth=1.5, zorder=3)

        delta = means[1] - means[0]
        sign  = "+" if delta >= 0 else "−"
        top   = max(means[0] + sems[0], means[1] + sems[1])
        y_ann = top + y_ann_base + 0.012
        ax.annotate(
            f"{group[:2]}: Δ={sign}{abs(delta):.3f}  {fmt_p(p)}",
            xy=(0.5, y_ann), ha="center", fontsize=5.6, color=gc)
        y_ann_base += max(sems) + 0.02

    ax.set_title(short_name(ct), fontsize=TITLE_FS, pad=3)
    ax.set_ylabel("Mean proportion", fontsize=LABEL_FS)
    ax.legend(fontsize=5.5, frameon=False, loc="upper left")
    style_ax(ax)

for ax in axes3_flat[n_types:]:
    ax.set_visible(False)

leg3 = [mlines.Line2D([], [], color=group_colors[g], linewidth=2.5, label=g)
        for g in groups]
fig3.legend(handles=leg3, fontsize=8, loc="lower right",
            frameon=False, title="Survivor group", title_fontsize=8.5)
fig3.suptitle(
    "Cohort-level Mean ± SEM — Long-term vs Short-term Survivors\n"
    "Primary vs Brain Met  |  Δ = met − primary",
    fontsize=10, fontweight="bold")

fig3.savefig("fig3_cohort_mean_sem-LTSvSTS.png",
             bbox_inches="tight", dpi=200)
fig3.savefig("fig3_cohort_mean_sem-LTSvSTS.pdf",
             bbox_inches="tight")
print("Figure 3 saved.")


# ══════════════════════════════════════════════════════════════════════════════
# FIGURE 4 — Mean ± SEM  (both groups overlaid, one panel per cell type)- select cell types
# ══════════════════════════════════════════════════════════════════════════════

import matplotlib.ticker as mticker

# figure 4 - selecting cell types

cell_types_of_interest = [
    "CEACAM-high tumor epithelial cells",
    "Mucin-producing tumor cells",
    "Cycling Tumor Cells",
]

selected_cell_types = [ct for ct in cell_types_of_interest if ct in cell_types]
n_types_sel = len(selected_cell_types)

ncols4 = 3
nrows4 = int(np.ceil(n_types_sel / ncols4))

fig4, axes4 = plt.subplots(
    nrows4, ncols4,
    figsize=(ncols4 * 4, nrows4 * 3.8),
    constrained_layout=True,
)

axes4_flat = axes4.flatten()

for idx, ct in enumerate(selected_cell_types):
    ax = axes4_flat[idx]

    y_ann_base = 0.0

    for g_idx, group in enumerate(groups):
        sub = df[(df["cell_type"] == ct) & (df["survivor_group"] == group)]
        if len(sub) < 2:
            continue

        # convert to percent
        primary = sub["primary_proportion"].values * 100
        met     = sub["brain_met_proportion"].values * 100

        gc = group_colors[group]
        p  = cohort_pvals[ct].get(group, np.nan)

        means = [np.mean(primary), np.mean(met)]
        sems  = [stats.sem(primary), stats.sem(met)]

        ax.fill_between(
            [0, 1],
            [means[0] - sems[0], means[1] - sems[1]],
            [means[0] + sems[0], means[1] + sems[1]],
            color=gc,
            alpha=0.15,
            zorder=1,
        )

        ax.plot(
            [0, 1],
            means,
            color=gc,
            linewidth=2.2,
            zorder=2,
            label=f"{group} (n={len(primary)})",
        )

        for xi, m, se in zip([0, 1], means, sems):
            ax.errorbar(
                xi,
                m,
                yerr=se,
                fmt="o",
                color=gc,
                markerfacecolor="white",
                markeredgewidth=1.6,
                markersize=8,
                capsize=5,
                capthick=1.5,
                linewidth=1.5,
                zorder=3,
            )

    ax.set_title(short_name(ct), fontsize=TITLE_FS, pad=4)

    # y-axis (bold + percent)
    ax.set_ylabel("Mean %", fontsize=LABEL_FS, fontweight="bold")
    #ax.yaxis.set_major_formatter(mticker.PercentFormatter())

    # x-axis (bold labels)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(
        ["Primary", "Brain Met"],
        fontsize=LABEL_FS,
        fontweight="bold",
    )

    style_ax(ax)

# hide empty panels
for ax in axes4_flat[n_types_sel:]:
    ax.set_visible(False)

# legend
leg4 = [
    mlines.Line2D([], [], color=group_colors[g], linewidth=2.5, label=g)
    for g in groups
]

fig4.legend(
    handles=leg4,
    fontsize=6,
    loc="upper right",
    frameon=False,
    title="Survivor group",
    title_fontsize=8.5,
)

fig4.suptitle(
    "Cohort-level Mean % ± SEM — Long-term vs Short-term Survivors\n"
    "Primary vs Brain Met | Δ = met − primary",
    fontsize=10,
    fontweight="bold",
)

fig4.savefig(
    "fig4_select-celltypes_mean_percent_sem-LTSvSTS.png",
    bbox_inches="tight",
    dpi=200,
)

fig4.savefig(
    "fig4_select-celltypes_mean_percent_sem-LTSvSTS.pdf",
    bbox_inches="tight",
)

print("Figure 4 saved.")


# ── P-value summary ────────────────────────────────────────────────────────────
print("\n── Wilcoxon p-values by cell type and survivor group ───────────────")
print(f"{'Cell type':<52} {'Long-term':>14}  {'Short-term':>14}")
print("─" * 84)
for ct in selected_cell_types:
    row = ""
    for g in groups:
        p = cohort_pvals[ct].get(g, np.nan)
        if np.isnan(p):
            row += f"  {'n/a':>12}"
        else:
            sig = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "ns"))
            row += f"  {p:>8.4f} {sig:>3}"
    print(f"{ct:<52}{row}")

# ── Mean summary ───────────────────────────────────────────────────────────────
print("\n── Mean % by cell type and survivor group ─────────────────────────")
print(
    f"{'Cell type':<40} "
    f"{'Group':<15} "
    f"{'Primary Mean %':>16} "
    f"{'Brain Met Mean %':>20}"
)
print("─" * 100)

for ct in selected_cell_types:
    for group in groups:

        sub = df[
            (df["cell_type"] == ct) &
            (df["survivor_group"] == group)
        ]

        if len(sub) < 2:
            print(
                f"{ct:<40} "
                f"{group:<15} "
                f"{'n/a':>16} "
                f"{'n/a':>20}"
            )
            continue

        primary_mean = sub["primary_proportion"].mean() * 100
        met_mean     = sub["brain_met_proportion"].mean() * 100

        print(
            f"{ct:<40} "
            f"{group:<15} "
            f"{primary_mean:>15.2f}% "
            f"{met_mean:>19.2f}%"
        )


In [ ]:

"""
Cell-type proportion plots — Malignant cells only — LTS vs STS
───────────────────────────────────────────────────────────────
Same three-figure layout as the all-cell script, but proportions are
calculated using only malignant cells:

    proportion = malignant_cell_type_count / total_malignant_cell_count

Two separate comparison sets:

  Set A  —  Primary tumours only:   Long-term vs Short-term survivors
  Set B  —  Brain mets only:        Long-term vs Short-term survivors

For each set, three figures are produced:

  Figure 1  →  Per-sample dot plot   (one panel per malignant cell type,
                                       jittered dots, diamond = mean ± SEM)
  Figure 2  →  Individual sample dots (LTS | STS side-by-side, same axes,
                                       no connecting lines, patient ID labels)
  Figure 3  →  Mean ± SEM summary    (LTS at x=0, STS at x=1, shared y-axis)

Statistics: Mann-Whitney U (unpaired, two-sided).

Expected input TSV columns:
  base_pid | cell_type | primary_count | brain_met_count
  (raw cell counts per cell type per patient; proportions are derived here)

Alternatively, if your TSV already has proportion columns, see the note
at the "Data" section below and adjust accordingly.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats

# ══════════════════════════════════════════════════════════════════════════════
# Data
# ══════════════════════════════════════════════════════════════════════════════
# Load raw counts (one row per patient × cell-type combination).
# Columns needed: base_pid, cell_type, primary_count, brain_met_count
# If your file already contains proportion columns computed over ALL cells,
# you will need raw counts (or a separate malignant-only file) to re-derive
# malignant-denominator proportions here.

df_raw = pd.read_csv(
    "primary_brain_met_proportions.tsv",
    sep="\t",
)

# ── Identify malignant cell types ──────────────────────────────────────────────
MALIGNANT_LABELS = [
    'CEACAM-high tumor epithelial cells',        # bright azure blue
    'Cycling Tumor Cells',                      # amber orange
    'Mucin-producing tumor cells', # — coral orange',       
    'Inflamed primary tumor epithelial cells',
]

is_malignant = df_raw["cell_type"].isin(MALIGNANT_LABELS)
df_mal = df_raw[is_malignant].copy()

if df_mal.empty:
    raise ValueError(
        "No rows matched MALIGNANT_LABELS. "
        f"Labels found in file: {sorted(df_raw['cell_type'].unique())}"
    )

# Warn if any expected label is missing from the data
found = set(df_mal["cell_type"].unique())
missing = [l for l in MALIGNANT_LABELS if l not in found]
if missing:
    print(f"  Warning: the following malignant labels were not found in the data:\n"
          + "\n".join(f"    • {l}" for l in missing))

# ── Compute malignant-denominator proportions ──────────────────────────────────
# The TSV contains proportions over ALL cells (primary_proportion,
# brain_met_proportion). Re-normalise within the malignant subset only:
#
#   mal_proportion = all_cell_proportion / sum(all_cell_proportions for
#                                              malignant types, per patient)
#
# This is mathematically identical to count / malignant_total when the
# all-cell proportions were derived from the same count denominator.

for all_prop_col, mal_prop_col in [
    ("primary_proportion",   "primary_mal_proportion"),
    ("brain_met_proportion", "brain_met_mal_proportion"),
]:
    denom_col = f"{mal_prop_col}_denom"
    mal_total = (
        df_mal.groupby("base_pid")[all_prop_col]
        .sum()
        .rename(denom_col)
    )
    df_mal = df_mal.join(mal_total, on="base_pid")
    df_mal[mal_prop_col] = df_mal[all_prop_col] / df_mal[denom_col]
    df_mal = df_mal.drop(columns=[denom_col])

# Drop rows where both re-normalised columns are NaN
df_mal = df_mal.dropna(
    subset=["primary_mal_proportion", "brain_met_mal_proportion"], how="all"
)

# ── Survivor classification ────────────────────────────────────────────────────
def classify_survivor(pid):
    try:
        n = int(pid.split("-")[1])
    except (IndexError, ValueError):
        return "unclassified"
    if 1 <= n <= 14:
        return "Long-term"
    if 45 <= n <= 65:
        return "Short-term"
    return "unclassified"

df_mal["survivor_group"] = df_mal["base_pid"].apply(classify_survivor)
df_mal = df_mal[df_mal["survivor_group"] != "unclassified"].copy()

# Working data frame (rename to df for shared helper functions)
df = df_mal.copy()

cell_types = sorted(df["cell_type"].unique())
n_types    = len(cell_types)

GROUPS       = ["Long-term", "Short-term"]
GROUP_COLORS = {"Long-term": "#2166AC", "Short-term": "#292929"}

# ══════════════════════════════════════════════════════════════════════════════
# Shared style helpers
# ══════════════════════════════════════════════════════════════════════════════
LABEL_FS = 14
TITLE_FS = 14
TICK_FS  = 14
DOT_S    = 28
JITTER_W = 0.12
RNG      = np.random.default_rng(42)

def style_ax_lts_sts(ax):
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Long-term", "Short-term"], fontsize=TICK_FS)
    ax.set_xlim(-0.55, 1.65)
    ax.tick_params(axis="y", labelsize=TICK_FS)
    for tick, grp in zip(ax.get_xticklabels(), GROUPS):
        tick.set_color(GROUP_COLORS[grp])
        tick.set_fontweight("bold")

def mannwhitney_p(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    a, b = a[~np.isnan(a)], b[~np.isnan(b)]
    if len(a) < 2 or len(b) < 2:
        return np.nan
    _, p = stats.mannwhitneyu(a, b, alternative="two-sided")
    return p

def fmt_p(p):
    if np.isnan(p):  return "P = n/a"
    if p < 0.001:    return "P < 0.001 ***"
    if p < 0.01:     return f"P = {p:.3f} **"
    if p < 0.05:     return f"P = {p:.3f} *"
    return f"P = {p:.2f} ns"

def short_name(ct):
    return ct.replace("(Cancer associated fibroblasts)", "(CAFs)")

def n_cols_rows(n, ncols=4):
    return ncols, int(np.ceil(n / ncols))

def significance_stars(p):
    """Return significance as asterisks only."""
    if np.isnan(p):
        return ""
    elif p < 0.0001:
        return "****"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return ""


# ══════════════════════════════════════════════════════════════════════════════
# Plotting functions
# ══════════════════════════════════════════════════════════════════════════════

def plot_fig1_dots(prop_col, location_label, file_stem, pval_store):
    """Figure 1 — Jittered dot plot, one panel per malignant cell type."""
    ncols, nrows = n_cols_rows(n_types)
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(ncols * 3.2, nrows * 3.8),
                             constrained_layout=True)
    axes_flat = np.array(axes).flatten()

    for idx, ct in enumerate(cell_types):
        ax  = axes_flat[idx]
        sub = df[df["cell_type"] == ct]

        lts_vals = sub.loc[sub["survivor_group"] == "Long-term",  prop_col].dropna().values
        sts_vals = sub.loc[sub["survivor_group"] == "Short-term", prop_col].dropna().values

        for xi, vals, grp in zip([0, 1], [lts_vals, sts_vals], GROUPS):
            gc     = GROUP_COLORS[grp]
            jitter = RNG.uniform(-JITTER_W, JITTER_W, size=len(vals))
            ax.scatter(xi + jitter, vals,
                       color="white", edgecolors=gc,
                       s=DOT_S, zorder=3, linewidths=0.9)
            if len(vals) > 0:
                m  = np.mean(vals)
                se = stats.sem(vals) if len(vals) > 1 else 0
                ax.errorbar(xi, m, yerr=se,
                            fmt="D", color=gc,
                            markerfacecolor=gc, markeredgewidth=1.2,
                            markersize=6, capsize=4, capthick=1.4,
                            linewidth=1.4, zorder=4)
            ax.text(xi, -0.04,
                    f"n={len(vals)}",
                    ha="center", va="top", fontsize=5.5, color="#777777",
                    transform=ax.get_xaxis_transform())

        p = mannwhitney_p(lts_vals, sts_vals)
        pval_store[ct] = p

        ax.set_title(f"{short_name(ct)}\n{fmt_p(p)}", fontsize=TITLE_FS, pad=3)
        ax.set_ylabel("Proportion of malignant cells", fontsize=LABEL_FS)
        style_ax_lts_sts(ax)

    for ax in axes_flat[n_types:]:
        ax.set_visible(False)

    handles = [mpatches.Patch(facecolor=GROUP_COLORS[g], label=g) for g in GROUPS]
    fig.legend(handles=handles, fontsize=7, loc="upper right",
               frameon=False, title="Survivor group", title_fontsize=7.5)
    fig.suptitle(
        f"Per-patient malignant cell-type proportions — {location_label}\n"
        "Long-term vs Short-term survivors  |  diamond = mean ± SEM  |  Mann-Whitney U\n"
        "Denominator = total malignant cells per patient",
        fontsize=10, fontweight="bold")

    fig.savefig(f"{file_stem}_mal_fig1_dots.png", bbox_inches="tight", dpi=200)
    fig.savefig(f"{file_stem}_mal_fig1_dots.pdf", bbox_inches="tight")
    plt.close(fig)
    print(f"  Figure 1 saved → {file_stem}_mal_fig1_dots")




def plot_fig2_sample_dots(prop_col, location_label, file_stem, pval_store):
    """Figure 2 — Individual sample dots, LTS & STS on same panel, patient ID labels."""

    cell_types_of_interest = [
        "CEACAM-high tumor epithelial cells",
        "Mucin-producing tumor cells",
        # "Cycling Tumor Cells",
        # "Inflamed primary tumor epithelial cells",
    ]

    selected_cell_types = [ct for ct in cell_types_of_interest if ct in cell_types]
    n_types_sel = len(selected_cell_types)

    ncols, nrows = n_cols_rows(n_types)
    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(ncols * 3.2, nrows * 3.8),
        constrained_layout=True,
    )

    axes_flat = np.array(axes).flatten()

    for idx, ct in enumerate(selected_cell_types):
        ax = axes_flat[idx]
        sub = df[df["cell_type"] == ct]
        p = pval_store.get(ct, np.nan)

        lts_sub = sub[sub["survivor_group"] == "Long-term"].dropna(subset=[prop_col])
        sts_sub = sub[sub["survivor_group"] == "Short-term"].dropna(subset=[prop_col])

        # convert to percent
        lts_vals = lts_sub[prop_col].values * 100
        sts_vals = sts_sub[prop_col].values * 100

        lts_pids = lts_sub["base_pid"].values
        sts_pids = sts_sub["base_pid"].values

        for xi, vals, pids, grp in zip(
            [0, 1],
            [lts_vals, sts_vals],
            [lts_pids, sts_pids],
            GROUPS,
        ):
            gc = GROUP_COLORS[grp]
            jitter = RNG.uniform(-JITTER_W, JITTER_W, size=len(vals))

            ax.scatter(
                xi + jitter,
                vals,
                color="white",
                edgecolors=gc,
                s=DOT_S,
                zorder=3,
                linewidths=0.9,
            )

            # mean ± SEM
            if len(vals) > 0:
                m = np.mean(vals)
                se = stats.sem(vals) if len(vals) > 1 else 0

                ax.errorbar(
                    xi,
                    m,
                    yerr=se,
                    fmt="D",
                    color=gc,
                    markerfacecolor=gc,
                    markeredgewidth=1.2,
                    markersize=6,
                    capsize=4,
                    capthick=1.4,
                    linewidth=1.4,
                    zorder=4,
                )

        stars = significance_stars(p)
        title_text = f"{short_name(ct)}"
        if stars:
            title_text += f"\n{stars}"

        ax.set_title(title_text, fontsize=TITLE_FS, pad=3)

        # y-axis in percent
        ax.set_ylabel("Mean %", fontsize=LABEL_FS, fontweight="bold",)

        style_ax_lts_sts(ax)

    # hide unused axes
    for ax in axes_flat[n_types:]:
        ax.set_visible(False)

    handles = [
        mpatches.Patch(facecolor=GROUP_COLORS[g], label=g)
        for g in GROUPS
    ]

    fig.legend(
        handles=handles,
        fontsize=7,
        loc="upper right",
        frameon=False,
        title="Survivor group",
        title_fontsize=7.5,
    )

    fig.suptitle(
        f"Individual malignant cell-type proportions — {location_label}\n"
        "Long-term vs Short-term survivors  |  diamond = mean ± SEM  |  Mann-Whitney U\n"
        "Denominator = total malignant cells per patient",
        fontsize=10,
        fontweight="bold",
    )

    fig.savefig(
        f"{file_stem}_mal_fig2_samples.png",
        bbox_inches="tight",
        dpi=200,
    )

    fig.savefig(
        f"{file_stem}_mal_fig2_samples.pdf",
        bbox_inches="tight",
    )

    plt.close(fig)
    print(f"  Figure 2 saved → {file_stem}_mal_fig2_samples")


def plot_fig3_mean_sem(prop_col, location_label, file_stem, pval_store):
    """Figure 3 — Mean ± SEM, LTS at x=0, STS at x=1, shared y-axis."""
    ncols, nrows = n_cols_rows(n_types)
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(ncols * 3.2, nrows * 3.6),
                             constrained_layout=True)
    axes_flat = np.array(axes).flatten()

    for idx, ct in enumerate(cell_types):
        ax = axes_flat[idx]
        p  = pval_store.get(ct, np.nan)

        lts_vals = df.loc[(df["cell_type"] == ct) &
                          (df["survivor_group"] == "Long-term"),
                          prop_col].dropna().values
        sts_vals = df.loc[(df["cell_type"] == ct) &
                          (df["survivor_group"] == "Short-term"),
                          prop_col].dropna().values

        upper_vals = []
        for xi, vals, grp in zip([0, 1], [lts_vals, sts_vals], GROUPS):
            if len(vals) < 1:
                continue
            gc = GROUP_COLORS[grp]
            m  = np.mean(vals)
            se = stats.sem(vals) if len(vals) > 1 else 0
            upper_vals.append(m + se)
            ax.errorbar(xi, m, yerr=se,
                        fmt="D", color=gc,
                        markerfacecolor=gc, markeredgewidth=1.2,
                        markersize=7, capsize=5, capthick=1.5,
                        linewidth=1.5, zorder=3)
            ax.text(xi, -0.04,
                    f"n={len(vals)}",
                    ha="center", va="top", fontsize=5.5, color="#777777",
                    transform=ax.get_xaxis_transform())

        if len(lts_vals) > 0 and len(sts_vals) > 0:
            delta = np.mean(sts_vals) - np.mean(lts_vals)
            sign  = "+" if delta >= 0 else "−"
            top   = max(upper_vals) if upper_vals else 0.05
            ax.annotate(
                f"Δ = {sign}{abs(delta):.3f}\n{fmt_p(p)}",
                xy=(0.5, top + 0.012),
                ha="center", fontsize=5.8, color="dimgray")

        ax.set_title(short_name(ct), fontsize=TITLE_FS, pad=3)
        ax.set_ylabel("Mean proportion of malignant cells", fontsize=LABEL_FS)
        style_ax_lts_sts(ax)

    for ax in axes_flat[n_types:]:
        ax.set_visible(False)

    handles = [mpatches.Patch(facecolor=GROUP_COLORS[g], label=g) for g in GROUPS]
    fig.legend(handles=handles, fontsize=7, loc="lower right",
               frameon=False, title="Survivor group", title_fontsize=7.5)
    fig.suptitle(
        f"Mean ± SEM malignant cell-type proportions — {location_label}\n"
        "Long-term vs Short-term survivors  |  Δ = Short-term − Long-term  |  Mann-Whitney U\n"
        "Denominator = total malignant cells per patient",
        fontsize=8, fontweight="bold")

    fig.savefig(f"{file_stem}_mal_fig3_mean_sem.png", bbox_inches="tight", dpi=200)
    fig.savefig(f"{file_stem}_mal_fig3_mean_sem.pdf", bbox_inches="tight")
    plt.close(fig)
    print(f"  Figure 3 saved → {file_stem}_mal_fig3_mean_sem")


def print_pval_summary(pval_store, location_label):
    print(f"\n── Mann-Whitney U p-values: {location_label} ───────────────────────")
    print(f"  {'Cell type':<52} {'P-value':>9}  Sig")
    print("  " + "─" * 68)
    for ct, p in pval_store.items():
        if np.isnan(p):
            print(f"  {ct:<52} {'n/a':>9}  —")
        else:
            sig = ("***" if p < 0.001 else "**" if p < 0.01
                   else "*" if p < 0.05 else "ns")
            print(f"  {ct:<52} {p:>9.4f}  {sig}")


# ══════════════════════════════════════════════════════════════════════════════
# RUN — two location sets
# ══════════════════════════════════════════════════════════════════════════════

SETS = [
    {
        "prop_col":       "primary_mal_proportion",
        "location_label": "Primary tumours (malignant cells only)",
        "file_stem":      "primary_lts_vs_sts",
    },
    {
        "prop_col":       "brain_met_mal_proportion",
        "location_label": "Brain metastases (malignant cells only)",
        "file_stem":      "brain_met_lts_vs_sts",
    },
]

for s in SETS:
    prop_col       = s["prop_col"]
    location_label = s["location_label"]
    file_stem      = s["file_stem"]

    print(f"\n{'═'*60}")
    print(f"  {location_label}  —  LTS vs STS")
    print(f"{'═'*60}")

    for g in GROUPS:
        pids = sorted(df[df["survivor_group"] == g]["base_pid"].unique())
        print(f"  {g} (n={len(pids)}): {', '.join(pids)}")

    print(f"\n  Malignant cell types included ({n_types}):")
    for ct in cell_types:
        print(f"    • {ct}")

    pval_store = {}

    plot_fig1_dots(prop_col, location_label, file_stem, pval_store)
    plot_fig2_sample_dots(prop_col, location_label, file_stem, pval_store)
    plot_fig3_mean_sem(prop_col, location_label, file_stem, pval_store)
    print_pval_summary(pval_store, location_label)

print("\nDone.")

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg") 
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
import scanpy as sc
from glob import glob

# ═════════════════════════════════════════════════════════════════════════════
# OUTPUT CONFIG
# ═════════════════════════════════════════════════════════════════════════════

OUTDIR = "docs/plots"
os.makedirs(OUTDIR, exist_ok=True)

# ═════════════════════════════════════════════════════════════════════════════
# CONSTANTS
# ═════════════════════════════════════════════════════════════════════════════

SAMPLE_COL      = "sample"
LOGNORM_LAYER   = "lognormal"
RAW_LAYER       = "counts"
IGNORE_PREFIX_N = 0

BRAIN_MET_LABELS = {"EAC Brain Met"}

MALIGNANT_LABELS = [
    'Mucin-producing tumor cells',
    'CEACAM-high tumor epithelial cells',
    'Inflamed primary tumor epithelial cells',
    'Cycling Tumor Cells',
]

CATEGORY_ORDER = ["ecDNA", "HSR", "copy number gain", "subclonal gain"]

CAT_STYLE = {
    "ecDNA": {"color": "#E05A2B", "marker": "^", "label": "ecDNA"},
    "HSR":   {"color": "#2B7BE0", "marker": "s", "label": "HSR"},
}

DOT_SIZE = 55
LABEL_FS = 14

cell_type_colors = {
    "CEACAM-high tumor epithelial cells": "#6BA4F8",
    "Cycling Tumor Cells": "#F4BA63",
    "Mucin-producing tumor cells": "#E59973",
    "Complement immunosuppressive macrophages (TAMs)": "#998CFA",
    "CAFs (Cancer associated fibroblasts)": "#9A5766",
    "Pericyte-enriched endothelial cells": "#4F709D",
    "Cytotoxic T cells": "#874284",
    "Plasma Cells": "#56B356",
}

# ═════════════════════════════════════════════════════════════════════════════
# HELPERS
# ═════════════════════════════════════════════════════════════════════════════

def normalize_acc1(s):
    if pd.isna(s):
        return ""
    s = str(s).strip().lower()
    return re.sub(r"[^\w\-]", "", s)


def lighten_color(color, amount=0.5):
    try:
        c = mcolors.to_rgb(color)
        white = (1, 1, 1)
        return tuple(c[i] + (white[i] - c[i]) * amount for i in range(3))
    except Exception:
        return color


def load_metadata(mapping_file, amp_file):
    mapping_df = pd.read_csv(mapping_file, dtype=str)
    mapping_df["Acc1_key"] = mapping_df["Acc1"].apply(normalize_acc1)

    amp_df = pd.read_csv(amp_file, dtype=str)
    amp_df["id_key"] = amp_df["ID"].str.strip().str.upper()
    amp_lookup = amp_df.set_index("id_key")["Amp_category"].to_dict()

    rows = []
    for _, r in mapping_df.iterrows():
        pid = None
        m = re.search(r"(P-\d+)", str(r.get("Updated_ID-Dec2025", "")))
        if m:
            pid = m.group(1)

        rows.append({
            "sample": r["Acc1"],
            "base_pid": pid,
            "amp_category": amp_lookup.get(pid, "No amp"),
            "Tumor Location": r.get("Tumor Location", "unknown"),
        })

    return pd.DataFrame(rows)


# ═════════════════════════════════════════════════════════════════════════════
# MAIN PIPELINE
# ═════════════════════════════════════════════════════════════════════════════

def run_pipeline(adata, mapping_file, amp_file):

    meta = load_metadata(mapping_file, amp_file)

    # ── filter brain mets ───────────────────────────────────────────────────
    meta = meta[meta["Tumor Location"].isin(BRAIN_MET_LABELS)]

    focal_samples = meta[meta["amp_category"].isin(["ecDNA", "HSR"])]["sample"]
    adata = adata[adata.obs[SAMPLE_COL].isin(focal_samples)].copy()

    adata = adata[adata.obs["cell_type"].isin(cell_type_colors)].copy()

    # ── compute proportions ─────────────────────────────────────────────────
    counts = adata.obs.groupby([SAMPLE_COL, "cell_type"]).size().unstack(fill_value=0)
    props = counts.div(counts.sum(axis=1), axis=0)

    df = (
        props.reset_index()
        .melt(id_vars=SAMPLE_COL, var_name="cell_type", value_name="prop")
        .merge(meta, on="sample")
    )

    # ── mean per category ───────────────────────────────────────────────────
    mean = (
        df.groupby(["amp_category", "cell_type"])["prop"]
        .mean()
        .unstack(fill_value=0)
        .reindex(["ecDNA", "HSR"])
    )

    # ═══════════════════════════════════════════════════════════════════════
    # FIGURE 1 — MEAN STACKED BAR
    # ═══════════════════════════════════════════════════════════════════════
    fig, ax = plt.subplots(figsize=(8, 4))

    mean.plot(
        kind="bar",
        stacked=True,
        ax=ax,
        color=[cell_type_colors.get(c, "#999") for c in mean.columns],
        edgecolor="black",
    )

    ax.set_ylabel("Mean cell fraction")
    ax.set_title("ecDNA vs HSR composition (brain mets)")
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)

    plt.tight_layout()

    fig.savefig(f"{OUTDIR}/mean_stacked.png", dpi=200)
    fig.savefig(f"{OUTDIR}/mean_stacked.svg")
    plt.close(fig)

    # ═══════════════════════════════════════════════════════════════════════
    # FIGURE 2 — HEATMAP-LIKE TABLE EXPORT (WEB FRIENDLY)
    # ═══════════════════════════════════════════════════════════════════════
    pivot = df.pivot_table(
        index="sample",
        columns="cell_type",
        values="prop",
        fill_value=0
    )

    pivot.to_csv(f"{OUTDIR}/celltype_proportions.csv")

    fig2, ax2 = plt.subplots(figsize=(10, 5))
    im = ax2.imshow(pivot.values, aspect="auto", cmap="viridis")

    ax2.set_xticks(range(len(pivot.columns)))
    ax2.set_xticklabels(pivot.columns, rotation=90)
    ax2.set_yticks(range(len(pivot.index)))
    ax2.set_yticklabels(pivot.index)

    plt.colorbar(im, ax=ax2, label="Cell fraction")

    plt.tight_layout()

    fig2.savefig(f"{OUTDIR}/heatmap.png", dpi=200)
    fig2.savefig(f"{OUTDIR}/heatmap.svg")
    plt.close(fig2)

    print(f"Saved outputs to {OUTDIR}")


# ═════════════════════════════════════════════════════════════════════════════
# ENTRY POINT
# ═════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":

    adata_path = "data/brain_met_adata.h5ad"
    mapping_file = "data/sample_mapping.csv"
    amp_file = "data/ID-by-ERBB2-amp-category.csv"

    adata = sc.read_h5ad(adata_path)

    run_pipeline(adata, mapping_file, amp_file)